In [2]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

# Local imports
from src.config import config
from src.utils import load_panel_data
from src.data_builder import DataBuilder

# Set monochrome publication style (black & white minimalist)
plt.rcParams.update({
    'figure.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'font.size': 11,
    'axes.edgecolor': 'black',
    'axes.facecolor': 'white',
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.grid': False,
    'grid.color': 'black',
    'grid.linestyle': ':',
})

# Directory structure (restructured)
output_dir = config.OUTPUT_DIR
tables_dir = output_dir / 'tables'
figures_dir = output_dir / 'figures'
agg_fig_dir = figures_dir / 'aggregate'
country_fig_dir = figures_dir / 'country'
for d in [tables_dir, agg_fig_dir, country_fig_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Imports & directories prepared:")
print(f"Tables -> {tables_dir}")
print(f"Aggregate Figures -> {agg_fig_dir}")
print(f"Country Figures -> {country_fig_dir}")

Imports & directories prepared:
Tables -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables
Aggregate Figures -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate
Country Figures -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/country


## 1. Load and Validate Data

In [3]:
# Load panel data
data_path = config.DATA_DIR / "df_panel_final.csv"

if not data_path.exists():
    print("Building dataset...")
    builder = DataBuilder()
    df_panel = builder.run()
else:
    df_panel = pd.read_csv(data_path, parse_dates=["date"])

# Country list matching legacy
COUNTRIES = {
    "Australia": "AUS",
    "Brazil": "BRA",
    "Canada": "CAN",
    "Euro Area": "EA",
    "Indonesia": "IND",
    "Japan": "JPN",
    "Korea": "KOR",
    "Mexico": "MEX",
    "New Zealand": "NZL",
    "Philippines": "PHL",
    "Switzerland": "CHE",
    "Thailand": "THA",
    "Türkiye": "TUR",
    "United Kingdom": "GBR"
}

print(f"Loaded data with {len(df_panel)} observations")
print(f"\nCountries in dataset:")
print(df_panel['country'].value_counts().sort_index())
print(f"\nDate range: {df_panel['date'].min()} to {df_panel['date'].max()}")

Loaded data with 4186 observations

Countries in dataset:
country
Australia         299
Brazil            299
Canada            299
Euro Area         299
Indonesia         299
Japan             299
Korea             299
Mexico            299
New Zealand       299
Philippines       299
Switzerland       299
Thailand          299
Türkiye           299
United Kingdom    299
Name: count, dtype: int64

Date range: 2000-02-29 00:00:00 to 2024-12-31 00:00:00


## 2. Descriptive Statistics Table

In [6]:
# Descriptive statistics (exchange rate returns + fundamental value)
# Fundamental value defined as log CPI differential: log(p_dom) - log(p_for)
from scipy import stats as sp_stats

stats_rows = []
for country_name in COUNTRIES.keys():
    sub = df_panel[df_panel['country'] == country_name].copy()
    if len(sub) == 0:
        continue
    r_s = pd.to_numeric(sub['r_s'], errors='coerce').dropna()  # exchange rate returns
    # fundamental value (log CPI differential)
    fv = (np.log(sub['p_dom']) - np.log(sub['p_for'])).dropna()
    stats_rows.append({
        'Country': country_name,
        'Mean_r_s': r_s.mean(),
        'Var_r_s': r_s.var(ddof=1),
        'Skew_r_s': sp_stats.skew(r_s),
        'Kurt_r_s': sp_stats.kurtosis(r_s),
        'Mean_f': fv.mean(),
        'Var_f': fv.var(ddof=1)
    })

df_desc = pd.DataFrame(stats_rows).set_index('Country')
print("\n=== DESCRIPTIVE STATISTICS (Returns & Fundamental Value) ===")
print(df_desc.round(4))

# Save CSV & LaTeX
csv_path = tables_dir / 'descriptive_statistics.csv'
tex_path = tables_dir / 'descriptive_statistics.tex'
df_desc.round(4).to_csv(csv_path)
with open(tex_path, 'w') as f:
    f.write(df_desc.round(4).to_latex(float_format=lambda x: f"{x:.4f}", caption='Descriptive Statistics: Exchange Rate Returns & Fundamental Value (f)', label='tab:desc_stats'))

print(f"Saved CSV -> {csv_path}")
print(f"Saved LaTeX -> {tex_path}")


=== DESCRIPTIVE STATISTICS (Returns & Fundamental Value) ===
                Mean_r_s  Var_r_s  Skew_r_s  Kurt_r_s  Mean_f   Var_f
Country                                                              
Australia         0.0001   0.0012    0.5630    2.0294  0.2773  0.1074
Brazil            0.0041   0.0023    0.8599    2.9949  1.0406  0.2732
Canada           -0.0000   0.0006    0.4183    2.5478 -0.2308  0.1242
Euro Area        -0.0002   0.0007    0.2075    1.3785 -0.0918  0.1642
Indonesia         0.0026   0.0010    0.0268    7.0811  1.2408  0.3364
Japan             0.0013   0.0007    0.1524    0.3847 -1.9950  0.5103
Korea             0.0009   0.0009    0.1386    5.1589  0.3488  0.4060
Mexico            0.0025   0.0011    1.3471    7.0182  0.6717  0.1294
New Zealand      -0.0004   0.0013    0.4077    0.5024  0.1383  0.2145
Philippines       0.0012   0.0003    0.8926    3.8278  0.7571  0.2318
Switzerland      -0.0020   0.0008   -0.1702    1.7484 -1.9071  1.4828
Thailand         -0.0003   0

## 3. STR Estimation Pipeline

### 3.1 Helper Functions for Z-Candidate Selection

In [7]:
# Z-candidate generation matching legacy
def build_z_candidates(data: pd.DataFrame) -> Dict[str, pd.Series]:
    """
    Generates 35 transition variable candidates:
    7 base variables × 5 lags each
    """
    z_base = {
        "eta": data["q"],
        "eta_abs": np.abs(data["q"]),
        "drs_abs": np.abs(data["r_s"]),
        "ID": (data["i_for"] - data["i_dom"]),
        "ppp_abs": np.abs(data["f_ppp"]),
        "drf_abs": np.abs(data["f_ppp_rel"]),
        "rel_misalignment_abs": np.abs(data["f_ppp_rel"] - data["r_s"]),
    }
    
    candidates = {}
    for name, series in z_base.items():
        for lag in range(1, 6):  # lags 1-5
            candidates[f"{name}_lag{lag}"] = series.shift(lag)
    
    return candidates

print("Z-candidate helper function defined")

Z-candidate helper function defined


### 3.2 LM Linearity Test Implementation

In [8]:
import statsmodels.api as sm
from scipy import stats

def lm_linearity_test_for_z(data: pd.DataFrame, z: pd.Series, maxlags_hac: int = 4) -> dict:
    """
    LM linearity test matching legacy implementation.
    Tests H0: Linear vs H1: STR for a specific z variable.
    """
    # Build base linear model
    df = pd.DataFrame({
        "y": data["r_s"] - (data["i_for"] - data["i_dom"]),
        "rs_lag1": data["r_s"].shift(1),
        "eta_lag1": data["q"].shift(1),
    }).dropna()
    
    # Join with z
    df = df.join(z.rename("z"), how="inner").dropna()
    
    if len(df) < 20:
        return None
    
    T = len(df)
    y = df["y"].to_numpy()
    X = df[["rs_lag1", "eta_lag1"]].to_numpy()
    
    # Linear model residuals
    lin_res = sm.OLS(y, X).fit()
    u = lin_res.resid
    
    # Taylor expansion terms
    zc = df["z"].to_numpy()
    z1, z2, z3 = zc, zc**2, zc**3
    
    W_rs = np.column_stack([
        df["rs_lag1"].to_numpy() * z1,
        df["rs_lag1"].to_numpy() * z2,
        df["rs_lag1"].to_numpy() * z3
    ])
    W_eta = np.column_stack([
        df["eta_lag1"].to_numpy() * z1,
        df["eta_lag1"].to_numpy() * z2,
        df["eta_lag1"].to_numpy() * z3
    ])
    W = np.column_stack([W_rs, W_eta])
    k = W.shape[1]
    
    # Project out X
    XTX_inv = np.linalg.pinv(X.T @ X)
    P_X = X @ XTX_inv @ X.T
    M_X = np.eye(T) - P_X
    W_star = M_X @ W
    
    # Auxiliary regression
    aux = sm.OLS(u, W_star).fit()
    R2 = aux.rsquared
    LM = T * R2
    pval = 1.0 - stats.chi2.cdf(LM, df=k)
    
    # HAC version
    aux_hac = sm.OLS(u, W_star).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags_hac})
    beta = np.asarray(aux_hac.params)
    cov = np.asarray(aux_hac.cov_params())
    cov_inv = np.linalg.pinv(cov)
    LM_HAC = float(beta.T @ cov_inv @ beta)
    pval_HAC = 1.0 - stats.chi2.cdf(LM_HAC, df=k)
    
    return {
        "LM": LM,
        "p_value": pval,
        "LM_HAC": LM_HAC,
        "p_value_HAC": pval_HAC,
        "df": k,
        "nobs": T
    }

print("LM linearity test function defined")

LM linearity test function defined


### 3.3 STR Estimation for All Countries

In [9]:
from src.econometrics import STRModel
from scipy.special import expit

# Storage for results
str_results = {}
str_linearity_tests = {}

for country_name, code in COUNTRIES.items():
    print(f"\n{'='*60}")
    print(f"Processing {country_name} ({code})")
    print(f"{'='*60}")
    
    try:
        # Get country data
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        
        if len(country_df) < 50:
            print(f"Insufficient data for {country_name}")
            continue
        
        # Generate z-candidates
        z_candidates = build_z_candidates(country_df)
        
        # Run linearity tests for all candidates
        print(f"Running linearity tests for {len(z_candidates)} candidates...")
        lm_results = []
        for z_name, z_series in z_candidates.items():
            result = lm_linearity_test_for_z(country_df, z_series, maxlags_hac=4)
            if result is not None:
                result['z_var'] = z_name
                lm_results.append(result)
        
        if len(lm_results) == 0:
            print(f"No valid linearity tests for {country_name}")
            continue
        
        lm_df = pd.DataFrame(lm_results).sort_values('p_value')
        str_linearity_tests[country_name] = lm_df
        
        # Select best z (lowest p-value)
        best_z_name = lm_df.iloc[0]['z_var']
        best_z = z_candidates[best_z_name]
        
        print(f"Best transition variable: {best_z_name} (p={lm_df.iloc[0]['p_value']:.4f})")
        
        # Build STR sample
        str_df = pd.DataFrame({
            "y": country_df["r_s"] - (country_df["i_for"] - country_df["i_dom"]),
            "rs_lag1": country_df["r_s"].shift(1),
            "eta_lag1": country_df["q"].shift(1),
            "z": best_z
        }).dropna()
        
        # Grid search with exponential gamma
        print(f"Running grid search...")
        gamma_grid = np.logspace(np.log10(0.125), np.log10(256.0), num=120)
        zc = pd.to_numeric(str_df["z"], errors="coerce").dropna().to_numpy()
        c_grid = np.quantile(zc, np.linspace(0.05, 0.95, 120))
        
        # Initialize and estimate
        str_model = STRModel(str_df, z_col='z')
        start_params = str_model.grid_search(gamma_grid, c_grid)
        
        print(f"Grid search complete. Starting NLS...")
        str_result = str_model.fit(start_params, hac_lags=4)
        
        # Store results
        str_results[country_name] = {
            'result': str_result,
            'z_name': best_z_name,
            'lm_pval': lm_df.iloc[0]['p_value']
        }
        
        print(f"\nSTR Results for {country_name}:")
        print(str_result.params)
        print(f"AIC: {str_result.aic:.2f}, BIC: {str_result.bic:.2f}")
        
    except Exception as e:
        print(f"ERROR processing {country_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\nSTR estimation complete for {len(str_results)} countries")


Processing Australia (AUS)
Running linearity tests for 35 candidates...
Best transition variable: eta_abs_lag4 (p=0.0035)
Running grid search...
Best transition variable: eta_abs_lag4 (p=0.0035)
Running grid search...
Grid search complete. Starting NLS...

STR Results for Australia:
const       0.003592
beta_c     -0.337267
beta_f      0.020466
gamma     561.099006
c           0.042039
const1     -0.003577
dtype: float64
AIC: -1991.67, BIC: -1969.55

Processing Brazil (BRA)
Running linearity tests for 35 candidates...
Best transition variable: eta_abs_lag2 (p=0.0000)
Running grid search...
Grid search complete. Starting NLS...

STR Results for Australia:
const       0.003592
beta_c     -0.337267
beta_f      0.020466
gamma     561.099006
c           0.042039
const1     -0.003577
dtype: float64
AIC: -1991.67, BIC: -1969.55

Processing Brazil (BRA)
Running linearity tests for 35 candidates...
Best transition variable: eta_abs_lag2 (p=0.0000)
Running grid search...
Grid search complete. S

## 4. BUIP Estimation for All Countries

### 3.4 STR Unrestricted Estimation for All Countries

In [10]:
from src.econometrics import STRModelNoRestrictions

# Storage for unrestricted STR results
str_unrest_results = {}

for country_name, code in COUNTRIES.items():
    print(f"\n{'='*60}")
    print(f"STR-Unrestricted: {country_name} ({code})")
    print(f"{'='*60}")
    
    try:
        # Get country data
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        
        if len(country_df) < 50:
            print(f"Insufficient data for {country_name}")
            continue
        
        # Use same transition variable as restricted STR
        if country_name not in str_results:
            print(f"Skipping {country_name}: No restricted STR result")
            continue
            
        best_z_name = str_results[country_name]['z_name']
        z_candidates = build_z_candidates(country_df)
        best_z = z_candidates[best_z_name]
        
        print(f"Using transition variable: {best_z_name}")
        
        # Build STR sample
        str_df = pd.DataFrame({
            "y": country_df["r_s"] - (country_df["i_for"] - country_df["i_dom"]),
            "rs_lag1": country_df["r_s"].shift(1),
            "eta_lag1": country_df["q"].shift(1),
            "z": best_z
        }).dropna()
        
        # Grid search with exponential gamma
        print(f"Running grid search...")
        gamma_grid = np.logspace(np.log10(0.125), np.log10(256.0), num=120)
        zc = pd.to_numeric(str_df["z"], errors="coerce").dropna().to_numpy()
        c_grid = np.quantile(zc, np.linspace(0.05, 0.95, 120))
        
        # Initialize and estimate
        str_unrest_model = STRModelNoRestrictions(str_df, z_col='z')
        start_params = str_unrest_model.grid_search(gamma_grid, c_grid)
        
        print(f"Grid search complete. Starting NLS...")
        str_unrest_result = str_unrest_model.fit(start_params, hac_lags=4)
        
        # Store results
        str_unrest_results[country_name] = {
            'result': str_unrest_result,
            'z_name': best_z_name
        }
        
        print(f"\nSTR-Unrestricted Results for {country_name}:")
        print(str_unrest_result.params)
        print(f"AIC: {str_unrest_result.aic:.2f}, BIC: {str_unrest_result.bic:.2f}")
        
    except Exception as e:
        print(f"ERROR processing {country_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\nSTR-Unrestricted estimation complete for {len(str_unrest_results)} countries")


STR-Unrestricted: Australia (AUS)
Using transition variable: eta_abs_lag4
Running grid search...
Grid search complete. Starting NLS...

STR-Unrestricted Results for Australia:
const0     -0.000222
betac0      0.120412
betaf0     -0.031255
gamma     256.000285
c           0.353429
const1      0.015543
betac1     -0.522237
betaf1     -0.006652
dtype: float64
AIC: -1995.10, BIC: -1965.61

STR-Unrestricted: Brazil (BRA)
Using transition variable: eta_abs_lag2
Running grid search...
Grid search complete. Starting NLS...

STR-Unrestricted Results for Brazil:
const0     0.010711
betac0     0.029019
betaf0    -0.010237
gamma     78.520551
c          0.549350
const1     0.484461
betac1    -1.388115
betaf1    -0.905688
dtype: float64
AIC: -1822.29, BIC: -1792.74

STR-Unrestricted: Canada (CAN)
Using transition variable: rel_misalignment_abs_lag2
Running grid search...
Grid search complete. Starting NLS...

STR-Unrestricted Results for Canada:
const0     -0.002545
betac0      0.262365
betaf0    

In [12]:
from src.econometrics import BUIPModel
from src.utils import build_beh_sample

# Storage for BUIP results
buip_results = {}

for country_name, code in COUNTRIES.items():
    print(f"\n{'='*60}")
    print(f"BUIP: {country_name} ({code})")
    print(f"{'='*60}")
    
    try:
        # Get country data
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        
        if len(country_df) < 50:
            print(f"Insufficient data for {country_name}")
            continue
        
        # Build BUIP sample
        buip_df = build_beh_sample(country_df)
        
        print(f"Sample size: {len(buip_df)}")
        
        # Estimate with multistart
        print(f"Running multistart optimization...")
        buip_model = BUIPModel(buip_df)
        buip_result = buip_model.fit_multistart(n_starts=None, hac_lags=4)
        
        # Store results
        buip_results[country_name] = buip_result
        
        print(f"\nBUIP Results for {country_name}:")
        print(buip_result.params)
        print(f"AIC: {buip_result.aic:.2f}, BIC: {buip_result.bic:.2f}")
        
    except Exception as e:
        print(f"ERROR processing {country_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\nBUIP estimation complete for {len(buip_results)} countries")


BUIP: Australia (AUS)
Sample size: 296
Running multistart optimization...

BUIP Results for Australia:
beta_f      0.020924
beta_c   -281.184115
gamma      48.346855
c         -32.762666
const     -69.643400
const1     69.646484
dtype: float64
AIC: -1996.83, BIC: -1974.69

BUIP: Brazil (BRA)
Sample size: 296
Running multistart optimization...

BUIP Results for Australia:
beta_f      0.020924
beta_c   -281.184115
gamma      48.346855
c         -32.762666
const     -69.643400
const1     69.646484
dtype: float64
AIC: -1996.83, BIC: -1974.69

BUIP: Brazil (BRA)
Sample size: 296
Running multistart optimization...

BUIP Results for Brazil:
beta_f      0.009956
beta_c     -0.328529
gamma     157.994187
c          -0.003535
const       0.017371
const1     -0.006447
dtype: float64
AIC: -1790.74, BIC: -1768.60

BUIP: Canada (CAN)
Sample size: 296
Running multistart optimization...

BUIP Results for Brazil:
beta_f      0.009956
beta_c     -0.328529
gamma     157.994187
c          -0.003535
const

## 5. Generate Publication Tables

In [13]:
def format_with_stars(estimate, pval):
    """Format estimates with stars and bold formatting for significance at 5% level."""
    if np.isnan(pval) or np.isnan(estimate):
        return ''
    val_str = f"{estimate:.4f}"
    if pval < 0.01:
        return f"\\textbf{{{val_str}}}***"
    if pval < 0.05:
        return f"\\textbf{{{val_str}}}**"
    if pval < 0.10:
        return f"{val_str}*"
    return val_str

# STR Results Table (CSV + LaTeX)
str_rows = []
for country_name in COUNTRIES.keys():
    if country_name in str_results:
        res = str_results[country_name]['result']
        row = {'Country': country_name}
        for param in ['const', 'beta_c', 'beta_f', 'gamma', 'c', 'const1']:
            est = res.params.get(param, np.nan)
            pval = res.pvals.get(param, np.nan)
            row[param] = format_with_stars(est, pval)
        str_rows.append(row)
    else:
        str_rows.append({'Country': country_name, 'const':'', 'beta_c':'', 'beta_f':'', 'gamma':'', 'c':'', 'const1':''})

df_str_table = pd.DataFrame(str_rows).set_index('Country')
print("\n=== STR MODEL RESULTS ===")
print(df_str_table)
str_csv = tables_dir / 'str_results_table.csv'
str_tex = tables_dir / 'str_results_table.tex'
df_str_table.to_csv(str_csv)
with open(str_tex, 'w') as f:
    f.write(df_str_table.to_latex(escape=False, caption='STR Model Estimates', label='tab:str_results'))
print(f"Saved STR CSV -> {str_csv}")
print(f"Saved STR LaTeX -> {str_tex}")

# BUIP Results Table (CSV + LaTeX)
buip_rows = []
for country_name in COUNTRIES.keys():
    if country_name in buip_results:
        res = buip_results[country_name]
        row = {'Country': country_name}
        for param in ['const', 'const1', 'beta_f', 'beta_c', 'gamma', 'c']:
            est = res.params.get(param, np.nan)
            pval = res.pvals.get(param, np.nan)
            row[param] = format_with_stars(est, pval)
        buip_rows.append(row)
    else:
        buip_rows.append({'Country': country_name, 'const':'', 'const1':'', 'beta_f':'', 'beta_c':'', 'gamma':'', 'c':''})

df_buip_table = pd.DataFrame(buip_rows).set_index('Country')
print("\n=== BUIP MODEL RESULTS ===")
print(df_buip_table)
buip_csv = tables_dir / 'buip_results_table.csv'
buip_tex = tables_dir / 'buip_results_table.tex'
df_buip_table.to_csv(buip_csv)
with open(buip_tex, 'w') as f:
    f.write(df_buip_table.to_latex(escape=False, caption='BUIP Model Estimates', label='tab:buip_results'))
print(f"Saved BUIP CSV -> {buip_csv}")
print(f"Saved BUIP LaTeX -> {buip_tex}")

print(f"Tables saved to {tables_dir}")


=== STR MODEL RESULTS ===
                             const              beta_c               beta_f  \
Country                                                                       
Australia                   0.0036  \textbf{-0.3373}**    \textbf{0.0205}**   
Brazil          \textbf{0.0135}***              0.0749    \textbf{0.0348}**   
Canada                     -0.0027  \textbf{0.2633}***               0.0178   
Euro Area        \textbf{0.0225}**             -0.1561              -0.0026   
Indonesia       \textbf{0.0062}***              0.0913              0.2924*   
Japan                      -0.0009              0.0585   \textbf{0.2106}***   
Korea                       0.0018              0.1225            2127.0964   
Mexico          \textbf{0.0088}***   \textbf{0.1611}**   \textbf{0.2510}***   
New Zealand                -0.0123             -0.0263             600.1606   
Philippines     \textbf{0.0026}***              0.0534               0.0504   
Switzerland     \textbf{-

In [14]:
# Aggregate figures (black & white minimalist)

# STR regime distribution
all_G_values = []
for country_name, result_dict in str_results.items():
    G = result_dict['result'].G.dropna()
    all_G_values.extend(G.values)

if all_G_values:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_G_values, bins=50, color='black', edgecolor='black', linewidth=0.8)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Transition Function G(z)')
    ax.set_ylabel('Frequency')
    ax.set_title('STR Regime Distribution')
    plt.tight_layout()
    fig_path_png = agg_fig_dir / 'str_regime_distribution.png'
    fig_path_pdf = agg_fig_dir / 'str_regime_distribution.pdf'
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved STR regime distribution -> {fig_path_pdf}")

# BUIP regime distribution
all_omega_values = []
for country_name, result in buip_results.items():
    omega = result.omega.dropna()
    all_omega_values.extend(omega.values)

if all_omega_values:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_omega_values, bins=50, color='black', edgecolor='black', linewidth=0.8)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Mixing Weight ω')
    ax.set_ylabel('Frequency')
    ax.set_title('BUIP Regime Distribution')
    plt.tight_layout()
    fig_path_png = agg_fig_dir / 'buip_regime_distribution.png'
    fig_path_pdf = agg_fig_dir / 'buip_regime_distribution.pdf'
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved BUIP regime distribution -> {fig_path_pdf}")

# Regime prevalence (STR vs BUIP)
str_chartist_pct = (np.array(all_G_values) > 0.5).mean() * 100 if all_G_values else 0
str_fund_pct = 100 - str_chartist_pct
buip_chartist_pct = (np.array(all_omega_values) < 0.5).mean() * 100 if all_omega_values else 0
buip_fund_pct = 100 - buip_chartist_pct

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, model, chartist, fund in [
    (axes[0], 'STR', str_chartist_pct, str_fund_pct),
    (axes[1], 'BUIP', buip_chartist_pct, buip_fund_pct)
]:
    ax.bar(['Chartist', 'Fundamentalist'], [chartist, fund], color='black', edgecolor='black', linewidth=0.9)
    ax.set_title(f'{model} Regime Prevalence')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Prevalence (%)') if ax is axes[0] else None
plt.tight_layout()
fig_path_png = agg_fig_dir / 'regime_prevalence_comparison.png'
fig_path_pdf = agg_fig_dir / 'regime_prevalence_comparison.pdf'
plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
plt.savefig(fig_path_pdf, bbox_inches='tight')
plt.close(fig)
print(f"Saved regime prevalence comparison -> {fig_path_pdf}")

Saved STR regime distribution -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/str_regime_distribution.pdf
Saved BUIP regime distribution -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/buip_regime_distribution.pdf
Saved BUIP regime distribution -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/buip_regime_distribution.pdf
Saved regime prevalence comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/regime_prevalence_comparison.pdf
Saved regime prevalence comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/regime_prevalence_comparison.pdf


In [16]:
# STR-Unrestricted Results Table
param_names_unrest = ['const0', 'betac0', 'betaf0', 'gamma', 'c', 'const1', 'betac1', 'betaf1']
records_unrest = []
for country_name, result_dict in str_unrest_results.items():
    res = result_dict['result']
    row = {'Country': country_name}
    for pname in param_names_unrest:
        est = res.params.get(pname, np.nan)
        se = res.se.get(pname, np.nan)
        pval = res.pvals.get(pname, np.nan)
        row[pname] = format_with_stars(est, pval)
        row[f'{pname}_se'] = f"({se:.4f})" if not np.isnan(se) else ""
    row['AIC'] = f"{res.aic:.2f}" if hasattr(res, 'aic') else ""
    row['BIC'] = f"{res.bic:.2f}" if hasattr(res, 'bic') else ""
    records_unrest.append(row)

df_unrest_table = pd.DataFrame(records_unrest)

# Interleave estimates and SEs
unrest_cols_ordered = ['Country']
for pname in param_names_unrest:
    unrest_cols_ordered.append(pname)
    unrest_cols_ordered.append(f'{pname}_se')
unrest_cols_ordered.extend(['AIC', 'BIC'])
df_unrest_table = df_unrest_table[unrest_cols_ordered]

# Save CSV
csv_path_unrest = tables_dir / 'str_unrestricted_results_table.csv'
df_unrest_table.to_csv(csv_path_unrest, index=False)
print(f"STR-Unrestricted results table saved -> {csv_path_unrest}")

# LaTeX with bold formatting
tex_path_unrest = tables_dir / 'str_unrestricted_results_table.tex'
with open(tex_path_unrest, 'w') as f:
    latex_str = df_unrest_table.to_latex(index=False, escape=False)
    f.write(latex_str)
print(f"STR-Unrestricted LaTeX table saved -> {tex_path_unrest}")
print(df_unrest_table.head())

STR-Unrestricted results table saved -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/str_unrestricted_results_table.csv
STR-Unrestricted LaTeX table saved -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/str_unrestricted_results_table.tex
     Country              const0 const0_se              betac0 betac0_se  \
0  Australia             -0.0002  (0.0020)              0.1204  (0.0839)   
1     Brazil  \textbf{0.0107}***  (0.0027)              0.0290  (0.0745)   
2     Canada             -0.0025  (0.0026)  \textbf{0.2624}***  (0.0954)   
3  Euro Area             0.0129*  (0.0070)            -0.1848*  (0.0971)   
4  Indonesia  \textbf{0.0055}***  (0.0014)              0.1114  (0.0748)   

                betaf0 betaf0_se                gamma      gamma_se  \
0   \textbf{-0.0313}**  (0.0127)             256.0003    (935.4209)   
1              -0.0102  (0.0110)  \textbf{78.5206}***     (19.3565)   
2              -0.0070  (0.0228)

## 7. Summary Report

In [17]:
print("\n" + "="*80)
print("REPRODUCTION PIPELINE COMPLETE")
print("="*80)

print(f"\nSTR Estimation:")
print(f"  - {len(str_results)} / {len(COUNTRIES)} countries estimated")
print(f"  - Results table: {tables_dir / 'str_results_table.csv'}")

print(f"\nSTR-Unrestricted Estimation:")
print(f"  - {len(str_unrest_results)} / {len(COUNTRIES)} countries estimated")
print(f"  - Results table: {tables_dir / 'str_unrestricted_results_table.csv'}")

print(f"\nBUIP Estimation:")
print(f"  - {len(buip_results)} / {len(COUNTRIES)} countries estimated")
print(f"  - Results table: {tables_dir / 'buip_results_table.csv'}")

print(f"\nOutput Files Generated:")
print(f"  - Descriptive statistics: {tables_dir / 'descriptive_statistics.csv'}")
print(f"  - STR results: {tables_dir / 'str_results_table.csv'}")
print(f"  - STR-Unrestricted results: {tables_dir / 'str_unrestricted_results_table.csv'}")
print(f"  - BUIP results: {tables_dir / 'buip_results_table.csv'}")
print(f"  - STR regime distribution: {agg_fig_dir / 'str_regime_distribution.pdf'}")
print(f"  - STR-Unrestricted regime distribution: {agg_fig_dir / 'str_unrestricted_regime_distribution.pdf'}")
print(f"  - BUIP regime distribution: {agg_fig_dir / 'buip_regime_distribution.pdf'}")
print(f"  - Regime prevalence comparison (3-way): {agg_fig_dir / 'regime_prevalence_comparison_all_models.pdf'}")

print(f"\nAll files saved to: {output_dir}")
print("\n" + "="*80)


REPRODUCTION PIPELINE COMPLETE

STR Estimation:
  - 14 / 14 countries estimated
  - Results table: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/str_results_table.csv

STR-Unrestricted Estimation:
  - 14 / 14 countries estimated
  - Results table: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/str_unrestricted_results_table.csv

BUIP Estimation:
  - 14 / 14 countries estimated
  - Results table: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/buip_results_table.csv

Output Files Generated:
  - Descriptive statistics: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/descriptive_statistics.csv
  - STR results: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/str_results_table.csv
  - STR-Unrestricted results: /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/tables/str_unrestricted_results_table.csv
  - BUIP results: /Users/lollo/Docum

In [18]:
# Detailed country plots (STR & BUIP) saved into figures/country
import importlib
import src.output_figures
importlib.reload(src.output_figures)
from src.output_figures import plot_country_detailed_analysis

print("\n==============================")
print("GENERATING DETAILED COUNTRY PLOTS")
print("==============================")

# STR detailed plots
for country_name in str_results.keys():
    try:
        result_dict = str_results[country_name]
        str_result = result_dict['result']
        z_name = result_dict['z_name']
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        z_candidates = build_z_candidates(country_df)
        z_series = z_candidates[z_name]
        y = country_df['r_s'] - (country_df['i_for'] - country_df['i_dom'])
        save_path = country_fig_dir / f"{country_name.replace(' ', '_')}_str_detailed.pdf"
        plot_country_detailed_analysis(
            y=y,
            transition_var=z_series,
            transition_func=str_result.G,
            c_threshold=str_result.params['c'],
            country=country_name,
            var_name=z_name,
            model_type='STR',
            save_path=save_path
        )
        print(f"✓ STR plot saved: {save_path.name}")
    except Exception as e:
        print(f"✗ STR {country_name}: {e}")

# BUIP detailed plots
for country_name in buip_results.keys():
    try:
        buip_result = buip_results[country_name]
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        buip_df = build_beh_sample(country_df)
        y = buip_df['y']
        utility_diff = buip_result.U_f - buip_result.U_c
        save_path = country_fig_dir / f"{country_name.replace(' ', '_')}_buip_detailed.pdf"
        plot_country_detailed_analysis(
            y=y,
            transition_var=utility_diff,
            transition_func=buip_result.omega,
            c_threshold=buip_result.params['c'],
            country=country_name,
            var_name='Utility Differential',
            model_type='BUIP',
            save_path=save_path
        )
        print(f"✓ BUIP plot saved: {save_path.name}")
    except Exception as e:
        print(f"✗ BUIP {country_name}: {e}")

print("\nDetailed country plots completed.")


GENERATING DETAILED COUNTRY PLOTS
✓ STR plot saved: Australia_str_detailed.pdf
✓ STR plot saved: Brazil_str_detailed.pdf
✓ STR plot saved: Australia_str_detailed.pdf
✓ STR plot saved: Brazil_str_detailed.pdf
✓ STR plot saved: Canada_str_detailed.pdf
✓ STR plot saved: Euro_Area_str_detailed.pdf
✓ STR plot saved: Canada_str_detailed.pdf
✓ STR plot saved: Euro_Area_str_detailed.pdf
✓ STR plot saved: Indonesia_str_detailed.pdf
✓ STR plot saved: Indonesia_str_detailed.pdf
✓ STR plot saved: Japan_str_detailed.pdf
✓ STR plot saved: Japan_str_detailed.pdf
✓ STR plot saved: Korea_str_detailed.pdf
✓ STR plot saved: Korea_str_detailed.pdf
✓ STR plot saved: Mexico_str_detailed.pdf
✓ STR plot saved: Mexico_str_detailed.pdf
✓ STR plot saved: New_Zealand_str_detailed.pdf
✓ STR plot saved: New_Zealand_str_detailed.pdf
✓ STR plot saved: Philippines_str_detailed.pdf
✓ STR plot saved: Philippines_str_detailed.pdf
✓ STR plot saved: Switzerland_str_detailed.pdf
✓ STR plot saved: Switzerland_str_detailed.p

In [19]:
# STR-Unrestricted detailed plots
for country_name in str_unrest_results.keys():
    try:
        result_dict = str_unrest_results[country_name]
        str_unrest_result = result_dict['result']
        z_name = result_dict['z_name']
        country_df = df_panel[df_panel['country'] == country_name].copy().set_index('date').sort_index()
        z_candidates = build_z_candidates(country_df)
        z_series = z_candidates[z_name]
        y = country_df['r_s'] - (country_df['i_for'] - country_df['i_dom'])
        save_path = country_fig_dir / f"{country_name.replace(' ', '_')}_str_unrestricted_detailed.pdf"
        plot_country_detailed_analysis(
            y=y,
            transition_var=z_series,
            transition_func=str_unrest_result.G,
            c_threshold=str_unrest_result.params['c'],
            country=country_name,
            var_name=z_name,
            model_type='STR-Unrestricted',
            save_path=save_path
        )
        print(f"✓ STR-Unrestricted plot saved: {save_path.name}")
    except Exception as e:
        print(f"✗ STR-Unrestricted {country_name}: {e}")

print("\nSTR-Unrestricted detailed country plots completed.")

✓ STR-Unrestricted plot saved: Australia_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Brazil_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Canada_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Euro_Area_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Indonesia_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Japan_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Korea_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Mexico_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: New_Zealand_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Philippines_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Switzerland_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Thailand_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: Türkiye_str_unrestricted_detailed.pdf
✓ STR-Unrestricted plot saved: United_Kingdom_str_unrestricted_detailed.pdf

STR-Unrestrict

In [20]:
# STR-Unrestricted regime distribution
all_G_unrest_values = []
for country_name, result_dict in str_unrest_results.items():
    G_unrest = result_dict['result'].G.dropna()
    all_G_unrest_values.extend(G_unrest.values)

if all_G_unrest_values:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(all_G_unrest_values, bins=50, color='black', edgecolor='black', linewidth=0.8)
    ax.axvline(0.5, color='red', linestyle='--', linewidth=1)
    ax.set_xlabel('Transition Function G(z)')
    ax.set_ylabel('Frequency')
    ax.set_title('STR-Unrestricted Regime Distribution')
    plt.tight_layout()
    fig_path_png = agg_fig_dir / 'str_unrestricted_regime_distribution.png'
    fig_path_pdf = agg_fig_dir / 'str_unrestricted_regime_distribution.pdf'
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved STR-Unrestricted regime distribution -> {fig_path_pdf}")

# Regime prevalence comparison (STR, STR-Unrest, BUIP)
str_chartist_pct = (np.array(all_G_values) > 0.5).mean() * 100 if all_G_values else 0
str_fund_pct = 100 - str_chartist_pct
str_unrest_chartist_pct = (np.array(all_G_unrest_values) > 0.5).mean() * 100 if all_G_unrest_values else 0
str_unrest_fund_pct = 100 - str_unrest_chartist_pct
buip_chartist_pct = (np.array(all_omega_values) < 0.5).mean() * 100 if all_omega_values else 0
buip_fund_pct = 100 - buip_chartist_pct

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, model, chartist, fund in [
    (axes[0], 'STR', str_chartist_pct, str_fund_pct),
    (axes[1], 'STR-Unrest', str_unrest_chartist_pct, str_unrest_fund_pct),
    (axes[2], 'BUIP', buip_chartist_pct, buip_fund_pct)
]:
    ax.bar(['Chartist', 'Fundamentalist'], [chartist, fund], color='black', edgecolor='black', linewidth=0.9)
    ax.set_title(f'{model} Regime Prevalence')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Prevalence (%)') if ax is axes[0] else None
plt.tight_layout()
fig_path_png = agg_fig_dir / 'regime_prevalence_comparison_all_models.png'
fig_path_pdf = agg_fig_dir / 'regime_prevalence_comparison_all_models.pdf'
plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
plt.savefig(fig_path_pdf, bbox_inches='tight')
plt.close(fig)
print(f"Saved 3-way regime prevalence comparison -> {fig_path_pdf}")

Saved STR-Unrestricted regime distribution -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/str_unrestricted_regime_distribution.pdf
Saved 3-way regime prevalence comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/regime_prevalence_comparison_all_models.pdf


## 8. Additional Publication-Quality Figures

In [21]:
# Gamma comparison (black & white) & RMSE comparison
print("\n==============================")
print("GAMMA & RMSE COMPARISONS")
print("==============================")

# Collect gamma values
str_gamma = {c: res['result'].params.get('gamma', np.nan) for c, res in str_results.items()}
str_unrest_gamma = {c: res['result'].params.get('gamma', np.nan) for c, res in str_unrest_results.items()}
buip_gamma = {c: res.params.get('gamma', np.nan) for c, res in buip_results.items()}

# Plot gamma comparison STR
if str_gamma:
    items = sorted(str_gamma.items(), key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i[0] for i in items], [i[1] for i in items], color='black', edgecolor='black', linewidth=0.9)
    ax.set_ylabel('Gamma')
    ax.set_title('STR Gamma by Country')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    path_pdf = agg_fig_dir / 'str_gamma_comparison.pdf'
    path_png = agg_fig_dir / 'str_gamma_comparison.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved STR gamma comparison -> {path_pdf}")

# Plot gamma comparison STR-Unrestricted
if str_unrest_gamma:
    items = sorted(str_unrest_gamma.items(), key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i[0] for i in items], [i[1] for i in items], color='black', edgecolor='black', linewidth=0.9)
    ax.set_ylabel('Gamma')
    ax.set_title('STR-Unrestricted Gamma by Country')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    path_pdf = agg_fig_dir / 'str_unrestricted_gamma_comparison.pdf'
    path_png = agg_fig_dir / 'str_unrestricted_gamma_comparison.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved STR-Unrestricted gamma comparison -> {path_pdf}")

# Plot gamma comparison BUIP
if buip_gamma:
    items = sorted(buip_gamma.items(), key=lambda x: x[1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i[0] for i in items], [i[1] for i in items], color='black', edgecolor='black', linewidth=0.9)
    ax.set_ylabel('Gamma')
    ax.set_title('BUIP Gamma by Country')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    path_pdf = agg_fig_dir / 'buip_gamma_comparison.pdf'
    path_png = agg_fig_dir / 'buip_gamma_comparison.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved BUIP gamma comparison -> {path_pdf}")

# RMSE model comparison (STR vs STR-Unrestricted vs BUIP)
comparison_rows = []
for c in COUNTRIES.keys():
    str_rmse = str_results.get(c, {}).get('result', None)
    str_rmse_val = str_rmse.rmse if str_rmse else np.nan
    str_unrest_rmse = str_unrest_results.get(c, {}).get('result', None)
    str_unrest_rmse_val = str_unrest_rmse.rmse if str_unrest_rmse else np.nan
    buip_res = buip_results.get(c, None)
    buip_rmse_val = buip_res.rmse if buip_res else np.nan
    if not np.isnan(str_rmse_val) or not np.isnan(str_unrest_rmse_val) or not np.isnan(buip_rmse_val):
        comparison_rows.append({
            'Country': c, 
            'STR_RMSE': str_rmse_val, 
            'STR_Unrest_RMSE': str_unrest_rmse_val,
            'BUIP_RMSE': buip_rmse_val
        })

if comparison_rows:
    df_rmse = pd.DataFrame(comparison_rows).set_index('Country')
    fig, ax = plt.subplots(figsize=(10, 4.5))
    x = np.arange(len(df_rmse.index))
    w = 0.25
    ax.bar(x - w, df_rmse['STR_RMSE'], width=w, color='black', edgecolor='black', label='STR')
    ax.bar(x, df_rmse['STR_Unrest_RMSE'], width=w, color='gray', edgecolor='black', label='STR-Unrest')
    ax.bar(x + w, df_rmse['BUIP_RMSE'], width=w, color='darkgray', edgecolor='black', label='BUIP')
    ax.set_xticks(x)
    ax.set_xticklabels(df_rmse.index, rotation=45)
    ax.set_ylabel('RMSE')
    ax.set_title('Model Comparison (RMSE)')
    ax.legend(frameon=False)
    plt.tight_layout()
    path_pdf = agg_fig_dir / 'model_rmse_comparison_all.pdf'
    path_png = agg_fig_dir / 'model_rmse_comparison_all.png'
    plt.savefig(path_pdf, bbox_inches='tight')
    plt.savefig(path_png, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved 3-way RMSE model comparison -> {path_pdf}")


GAMMA & RMSE COMPARISONS
Saved STR gamma comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/str_gamma_comparison.pdf
Saved STR gamma comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/str_gamma_comparison.pdf
Saved STR-Unrestricted gamma comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/str_unrestricted_gamma_comparison.pdf
Saved STR-Unrestricted gamma comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/str_unrestricted_gamma_comparison.pdf
Saved BUIP gamma comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/buip_gamma_comparison.pdf
Saved BUIP gamma comparison -> /Users/lollo/Documents/Current_Projects/str-buip-estimation/results/figures/aggregate/buip_gamma_comparison.pdf
Saved 3-way RMSE model comparison -> /Users/lollo/Documents/Current_Proj

## Final Summary

In [22]:
print("\n" + "="*80)
print("PIPELINE COMPLETE (CURATED OUTPUT)")
print("="*80)

print(f"STR models estimated: {len(str_results)}")
print(f"STR-Unrestricted models estimated: {len(str_unrest_results)}")
print(f"BUIP models estimated: {len(buip_results)}")

print("\nTables (CSV + LaTeX):")
for p in ['descriptive_statistics', 'str_results_table', 'str_unrestricted_results_table', 'buip_results_table']:
    print(f"  - {p}.csv / {p}.tex")

print("\nAggregate Figures:")
for p in ['str_regime_distribution', 'str_unrestricted_regime_distribution', 'buip_regime_distribution', 
          'regime_prevalence_comparison_all_models', 'str_gamma_comparison', 'str_unrestricted_gamma_comparison',
          'buip_gamma_comparison', 'model_rmse_comparison_all']:
    print(f"  - {p}.pdf / {p}.png")

print("\nCountry Figures (Detailed Plots):")
print(f"  - {len([f for f in country_fig_dir.glob('*_str_detailed.pdf')])} STR detailed plots")
print(f"  - {len([f for f in country_fig_dir.glob('*_str_unrestricted_detailed.pdf')])} STR-Unrestricted detailed plots")
print(f"  - {len([f for f in country_fig_dir.glob('*_buip_detailed.pdf')])} BUIP detailed plots")

print("\nDirectory Structure:")
print(f"  Tables -> {tables_dir}")
print(f"  Figures (Aggregate) -> {agg_fig_dir}")
print(f"  Figures (Country)  -> {country_fig_dir}")

print("\nRetained components match user specification.")
print("="*80)


PIPELINE COMPLETE (CURATED OUTPUT)
STR models estimated: 14
STR-Unrestricted models estimated: 14
BUIP models estimated: 14

Tables (CSV + LaTeX):
  - descriptive_statistics.csv / descriptive_statistics.tex
  - str_results_table.csv / str_results_table.tex
  - str_unrestricted_results_table.csv / str_unrestricted_results_table.tex
  - buip_results_table.csv / buip_results_table.tex

Aggregate Figures:
  - str_regime_distribution.pdf / str_regime_distribution.png
  - str_unrestricted_regime_distribution.pdf / str_unrestricted_regime_distribution.png
  - buip_regime_distribution.pdf / buip_regime_distribution.png
  - regime_prevalence_comparison_all_models.pdf / regime_prevalence_comparison_all_models.png
  - str_gamma_comparison.pdf / str_gamma_comparison.png
  - str_unrestricted_gamma_comparison.pdf / str_unrestricted_gamma_comparison.png
  - buip_gamma_comparison.pdf / buip_gamma_comparison.png
  - model_rmse_comparison_all.pdf / model_rmse_comparison_all.png

Country Figures (Detail